# Notebook 04 — Lógica Paraconsistente aplicada à base de diabetes

Neste notebook, vamos trabalhar com conflito de informação.

Em bases clínicas reais, sistemas diferentes podem discordar.  
Uma fonte pode dizer que um sintoma está presente, enquanto outra diz que não.

A lógica clássica tem dificuldade com contradições.  
A lógica paraconsistente permite continuar raciocinando sem fazer a inferência colapsar.

A ideia principal é:

> Diante de conflito, o sistema deve sinalizar a inconsistência, e não concluir qualquer coisa.

## Etapa 1 — Enviar o arquivo da base

In [ ]:
# ============================================================
# 1. Faça upload do arquivo diabetes_data_upload.csv
# ============================================================

from google.colab import files
import io
import pandas as pd

print("Selecione o arquivo diabetes_data_upload.csv no seu computador.")
uploaded = files.upload()

# Pega automaticamente o primeiro arquivo enviado
nome_arquivo = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]))

print("Arquivo carregado com sucesso!")
print(f"Nome do arquivo: {nome_arquivo}")
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

df.head()

## Etapa 2 — Preparar funções auxiliares

In [ ]:
# ============================================================
# Bibliotecas e funções auxiliares
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Coluna-alvo da base
TARGET = "class"

# Marcadores clínico-sintomáticos presentes na base
SINTOMAS = [
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "Genital thrush",
    "visual blurring",
    "Itching",
    "Irritability",
    "delayed healing",
    "partial paresis",
    "muscle stiffness",
    "Alopecia",
    "Obesity",
]

# Sintomas escolhidos como centrais para simular lacunas ou conflito
# A escolha é didática: são sintomas que aparecem com relevância clínica
# e ajudam a mostrar o comportamento das lógicas.
SINTOMAS_CENTRAIS = [
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "partial paresis",
]

# Pesos heurísticos.
# Eles não foram aprendidos por machine learning.
# São apenas uma forma didática de dizer:
# "alguns sintomas pesam mais na suspeita do que outros".
PESOS = {
    "Polyuria": 3.0,
    "Polydipsia": 3.0,
    "sudden weight loss": 2.0,
    "weakness": 1.0,
    "Polyphagia": 1.5,
    "Genital thrush": 0.5,
    "visual blurring": 1.0,
    "Itching": 0.25,
    "Irritability": 1.0,
    "delayed healing": 0.75,
    "partial paresis": 2.0,
    "muscle stiffness": 0.5,
    "Alopecia": 0.25,
    "Obesity": 0.25,
}

def sim_nao_para_numero(valor):
    '''
    Converte respostas categóricas da base para números.

    Yes -> 1
    No  -> 0
    NaN -> NaN

    Essa conversão permite que a lógica opere sobre evidências.
    '''
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().lower()

    if valor == "yes":
        return 1.0
    elif valor == "no":
        return 0.0
    else:
        return np.nan

def pertinencia_idade(idade):
    '''
    Transforma idade em uma evidência gradual simples.

    Menos de 30 anos: pertinência 0
    Entre 30 e 60 anos: crescimento linear
    Acima de 60 anos: pertinência 1

    Não é uma regra médica definitiva.
    É apenas uma simplificação didática para mostrar como
    variáveis numéricas podem entrar no raciocínio lógico.
    '''
    if pd.isna(idade):
        return 0.5

    if idade < 30:
        return 0.0
    elif idade > 60:
        return 1.0
    else:
        return (idade - 30) / 30

def classe_para_binario(valor):
    '''
    Converte a classe original da base para comparação simples.

    Positive -> 1
    Negative -> 0
    '''
    return 1 if valor == "Positive" else 0

def mostrar_distribuicao_decisoes(resultado, coluna="decisao", titulo="Distribuição das decisões"):
    '''
    Plota um gráfico simples com a distribuição das saídas produzidas pela lógica.
    '''
    contagem = resultado[coluna].value_counts()

    plt.figure(figsize=(8, 4))
    barras = plt.bar(contagem.index.astype(str), contagem.values)
    plt.title(titulo)
    plt.xlabel("Saída produzida pela lógica")
    plt.ylabel("Quantidade de registros")
    plt.xticks(rotation=25, ha="right")

    for barra in barras:
        altura = barra.get_height()
        plt.text(
            barra.get_x() + barra.get_width()/2,
            altura,
            str(int(altura)),
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

def resumo_simples(resultado, coluna_decisao="decisao"):
    '''
    Gera um resumo didático da saída da lógica.

    A ideia não é tratar a lógica como um classificador tradicional,
    mas observar como ela decide, quando se abstém e como distribui
    suas respostas.
    '''
    total = len(resultado)

    resumo = resultado[coluna_decisao].value_counts().reset_index()
    resumo.columns = ["decisao", "quantidade"]
    resumo["percentual"] = 100 * resumo["quantidade"] / total

    return resumo

## Etapa 3 — Conhecer a base

In [ ]:
# ============================================================
# Olhando a base antes de aplicar qualquer lógica
# ============================================================

print("Dimensão da base:", df.shape)
print("\nColunas disponíveis:")
print(df.columns.tolist())

print("\nDistribuição da classe:")
display(df[TARGET].value_counts().reset_index().rename(columns={"index": "classe", TARGET: "quantidade"}))

print("\nQuantidade de valores ausentes por coluna:")
display(df.isna().sum().reset_index().rename(columns={"index": "coluna", 0: "ausentes"}))

print("\nResumo da idade:")
display(df["Age"].describe())

## Etapa 4 — Criar duas fontes com possíveis conflitos

A base original tem uma única versão dos dados.

Para estudar lógica paraconsistente, vamos simular duas fontes:

- **Fonte A**: mantém os dados originais;
- **Fonte B**: inverte parte dos sintomas centrais.

Isso cria conflitos controlados.

In [ ]:
# ============================================================
# Criando duas fontes contraditórias
# ============================================================

def criar_fontes_contraditorias(base, taxa_conflito=0.20, semente=42):
    '''
    Cria duas versões da base:

    Fonte A: mantém os dados originais.
    Fonte B: inverte parte dos sintomas centrais.

    Exemplo:
    Se a fonte A diz Polyuria = Yes,
    a fonte B pode dizer Polyuria = No.

    Isso simula conflito entre fontes, prontuários, entrevistas
    ou sistemas diferentes.
    '''
    fonte_a = base.copy()
    fonte_b = base.copy()

    rng = np.random.default_rng(semente)

    for coluna in SINTOMAS_CENTRAIS:
        mascara = rng.random(len(fonte_b)) < taxa_conflito
        fonte_b.loc[mascara, coluna] = fonte_b.loc[mascara, coluna].map({
            "Yes": "No",
            "No": "Yes"
        })

    return fonte_a, fonte_b

fonte_a, fonte_b = criar_fontes_contraditorias(df, taxa_conflito=0.20)

print("Fonte A:")
display(fonte_a[SINTOMAS_CENTRAIS].head())

print("Fonte B:")
display(fonte_b[SINTOMAS_CENTRAIS].head())

## Etapa 5 — Ideia da lógica paraconsistente

Para cada registro, calcularemos:

- grau de evidência favorável;
- grau de evidência contrária;
- grau de certeza;
- quantidade de conflitos.

Se houver conflito relevante, a saída será `inconsistente`.

Isso não significa que o paciente está inconsistente.  
Significa que as fontes de informação sobre aquele registro estão em conflito.

In [ ]:
def decisao_paraconsistente(linha_a, linha_b=None):
    if linha_b is None:
        linha_b = linha_a

    evidencia_favoravel = 0
    evidencia_contraria = 0
    soma_pesos = 0

    conflitos = 0
    conflitos_centrais = 0

    for sintoma in SINTOMAS:
        valor_a = sim_nao_para_numero(linha_a[sintoma])
        valor_b = sim_nao_para_numero(linha_b[sintoma])
        peso = PESOS[sintoma]

        # Se as duas fontes não sabem, não há evidência.
        if pd.isna(valor_a) and pd.isna(valor_b):
            continue

        valores_disponiveis = [
            valor for valor in [valor_a, valor_b]
            if not pd.isna(valor)
        ]

        if len(valores_disponiveis) == 0:
            continue

        media_favoravel = np.mean(valores_disponiveis)
        media_contraria = 1 - media_favoravel

        evidencia_favoravel += media_favoravel * peso
        evidencia_contraria += media_contraria * peso
        soma_pesos += peso

        # Conflito explícito: uma fonte diz Yes e outra diz No.
        if len(valores_disponiveis) == 2 and valor_a != valor_b:
            conflitos += 1

            if sintoma in SINTOMAS_CENTRAIS:
                conflitos_centrais += 1

    # Idade entra como componente gradual complementar.
    idade_fav = pertinencia_idade(linha_a["Age"])
    evidencia_favoravel += idade_fav * 0.75
    evidencia_contraria += (1 - idade_fav) * 0.75
    soma_pesos += 0.75

    if soma_pesos == 0:
        return pd.Series({
            "grau_favoravel": np.nan,
            "grau_contrario": np.nan,
            "grau_certeza": np.nan,
            "conflitos": conflitos,
            "conflitos_centrais": conflitos_centrais,
            "decisao": "indeterminado"
        })

    grau_favoravel = evidencia_favoravel / soma_pesos
    grau_contrario = evidencia_contraria / soma_pesos
    grau_certeza = grau_favoravel - grau_contrario

    conflito_relevante = conflitos_centrais >= 1 or conflitos >= 3

    if conflito_relevante:
        decisao = "inconsistente"
    elif grau_certeza >= 0.30:
        decisao = "alto"
    elif grau_certeza <= -0.30:
        decisao = "baixo"
    else:
        decisao = "indeterminado"

    return pd.Series({
        "grau_favoravel": grau_favoravel,
        "grau_contrario": grau_contrario,
        "grau_certeza": grau_certeza,
        "conflitos": conflitos,
        "conflitos_centrais": conflitos_centrais,
        "decisao": decisao
    })

def aplicar_logica_paraconsistente(fonte_a, fonte_b=None):
    resultado = fonte_a.copy()

    if fonte_b is None:
        saidas = fonte_a.apply(lambda linha: decisao_paraconsistente(linha), axis=1)
    else:
        saidas = pd.DataFrame([
            decisao_paraconsistente(fonte_a.iloc[i], fonte_b.iloc[i])
            for i in range(len(fonte_a))
        ])

    resultado = pd.concat(
        [resultado.reset_index(drop=True), saidas.reset_index(drop=True)],
        axis=1
    )

    resultado["classe_binaria"] = resultado[TARGET].apply(lambda x: 1 if x == "Positive" else 0)
    return resultado

## Etapa 6 — Aplicar sem conflito externo

Primeiro, vamos aplicar a lógica considerando apenas uma fonte.

Nesse cenário, não há conflito entre fontes diferentes.

In [ ]:
resultado_sem_conflito = aplicar_logica_paraconsistente(df)

display(resultado_sem_conflito[[
    "Age", "Gender", "Polyuria", "Polydipsia",
    "class", "grau_favoravel", "grau_contrario",
    "grau_certeza", "conflitos", "decisao"
]].head(10))

display(resumo_simples(resultado_sem_conflito))
mostrar_distribuicao_decisoes(
    resultado_sem_conflito,
    titulo="Lógica paraconsistente — sem conflito externo"
)

## Etapa 7 — Aplicar com fontes contraditórias

Agora usamos a Fonte A e a Fonte B.

O objetivo é observar quando a lógica sinaliza `inconsistente`.

In [ ]:
resultado_com_conflito = aplicar_logica_paraconsistente(fonte_a, fonte_b)

display(resultado_com_conflito[[
    "Age", "Gender", "Polyuria", "Polydipsia",
    "class", "grau_favoravel", "grau_contrario",
    "grau_certeza", "conflitos", "conflitos_centrais", "decisao"
]].head(10))

display(resumo_simples(resultado_com_conflito))
mostrar_distribuicao_decisoes(
    resultado_com_conflito,
    titulo="Lógica paraconsistente — com fontes contraditórias"
)

## Etapa 8 — Reflexão

A saída `inconsistente` não é uma falha do algoritmo.

Ela é justamente o comportamento desejado quando há conflito informacional.

A lógica paraconsistente permite que um sistema de apoio à decisão diga:

> Existem evidências contraditórias. A decisão deve ser interpretada com cautela.

Isso é muito útil em ambientes clínicos com dados vindos de múltiplas fontes.

In [ ]:
resultado_sem_conflito.to_csv("resultado_paraconsistente_sem_conflito.csv", index=False)
resultado_com_conflito.to_csv("resultado_paraconsistente_com_conflito.csv", index=False)

print("Arquivos exportados:")
print("- resultado_paraconsistente_sem_conflito.csv")
print("- resultado_paraconsistente_com_conflito.csv")